In [1]:
# 构造一个“与 CT2 一致”的 encoder_output（bfloat16 StorageView, 形状 1x1491x1280），
# 然后转换为 Kimi 期望的张量（torch.bfloat16 CPU, 形状 1x1492x1280），最后给出 Kimi reshape 后的 1x373x5120。

import numpy as np
import torch

def make_ct2_encoder_output_like():
    import ctranslate2 as ct2
    rng = np.random.default_rng(0)
    arr = rng.standard_normal((1, 1491, 1280), dtype=np.float32)
    sv = ct2.StorageView.from_array(arr)
    sv = sv.to(ct2.DataType.bfloat16)  # CT2 实际是 bfloat16
    # 如需在 GPU 上模拟，可尝试移到 CUDA（若失败则保持 CPU）
    try:
        # 0=CPU，在某些版本中 1 可能表示 CUDA:0；若不支持会抛异常
        sv = sv.to_device(ct2.Device(1))
    except Exception:
        pass
    return sv

def ct2_to_kimi_tensor(sv):
    import ctranslate2 as ct2
    # 1) 移到 CPU 并转 float32，才能安全拿到 numpy 视图
    sv_cpu = sv.to_device(ct2.Device(0)).to(ct2.DataType.float32)
    arr = np.array(sv_cpu)
    if arr.dtype == object:
        arr = np.stack(arr)
    assert arr.shape == (1, 1491, 1280), f"unexpected shape from CT2: {arr.shape}"

    # 2) 帧数补到 4 的倍数（1491 -> 1492），Kimi 后续要做 4×拼接
    pad_len = (4 - (arr.shape[1] % 4)) % 4
    if pad_len:
        arr = np.pad(arr, ((0,0),(0,pad_len),(0,0)), mode="constant")  # 或 mode='edge' 复制最后一帧

    # 3) 转 torch.bfloat16（Kimi 侧打印的是 bfloat16, device=cpu）
    feat = torch.from_numpy(arr).to(torch.bfloat16).cpu()
    assert feat.shape == (1, 1492, 1280), feat.shape
    return feat

def kimi_reshape_4x(feat: torch.Tensor):
    # [B, T, 1280] -> [B, T//4, 5120]
    b, t, h = feat.shape
    assert t % 4 == 0 and h == 1280
    return feat.reshape(b, t // 4, h * 4)

# 生成“CT2 一样”的 encoder_output
encoder_output_like = make_ct2_encoder_output_like()
print("[CT2 mock] repr:", encoder_output_like)  # 通常会显示 “… storage viewed as 1x1491x1280 …”

# 转为 Kimi 期望的 whisper_feature
whisper_feature = ct2_to_kimi_tensor(encoder_output_like)
print("[Kimi] whisper_feature:", type(whisper_feature), whisper_feature.dtype, whisper_feature.device, whisper_feature.shape)

# （可选）做 Kimi 的 4×拼接 reshape
whisper_feature_4x = kimi_reshape_4x(whisper_feature)
print("[Kimi] reshaped:", whisper_feature_4x.shape)  # 期望 (1, 373, 5120)

[2025-09-20 06:25:47.876] [info] Using CUDA allocator: cuda_malloc_async
[CT2 mock] repr:  1.11719 -1.39062 -0.425781 ... -0.339844 -1.88281 1.75
[cuda:0 bfloat16 storage viewed as 1x1491x1280]
[Kimi] whisper_feature: <class 'torch.Tensor'> torch.bfloat16 cpu torch.Size([1, 1492, 1280])
[Kimi] reshaped: torch.Size([1, 373, 5120])


In [1]:
import soundfile as sf, numpy as np, librosa, requests

path = "/workspace/ASR/audio_examples/cantonese.wav"
wav, sr = sf.read(path, dtype="float32")
if wav.ndim > 1:
    wav = wav.mean(axis=1)
if sr != 16000:
    wav = librosa.resample(wav, orig_sr=sr, target_sr=16000)

data = wav.astype(np.float32).tobytes()
r = requests.post(
    "http://localhost:8001/transcribe_websocket",
    headers={"Content-Type": "application/octet-stream", "prep_timeout": "60"},
    data=data
)
print(r.json())

{'result': [{'status': 0, 'text': '「成日都覺得生活好忙,啲嘢一戰接一戰,根本無時間停低諗清楚,有時明知自己做咁嘅嘢唔啱,但又唔知點樣改,朋友提過我應該學下點樣放鬆,唔好成日咁緊張,不過講就已真係做到又係另一回事,啱啱見到街邊條貓心懶腰,我都想學佢咁自在。」 「生活真係要識得慢落嚟,唔好靜氣上前行。」', 'language': '<|yue|>', 'confidence': 0.999761164188385, 'engine': 'whisper', 'total_time': 1.2690987586975098, 'start': 0, 'end': 29.801125}], 'info': {'engine': 'whisper', 'language': '<|yue|>', 'confidence': 0.999761164188385}, 'status': 0}


In [5]:
# 简单连接测试: simple_test.py
!pip install websocket-client
import websocket._core as websocket
import json

def simple_test():

        ws = websocket.create_connection("ws://127.0.0.1:9092")
        print("✅ 连接成功！")
        
        # 发送初始化
        init_msg = {
            "uid": "test",
            "token": "test",
            "name": "test.wav",
            "model": "faster_whisper"
        }
        ws.send(json.dumps(init_msg))
        print("✅ 初始化成功！")
        
        # 尝试接收响应
        response = ws.recv()
        print(f"📨 服务器响应: {response}")
        
        ws.close()
        print("✅ 测试完成！")
        


if __name__ == "__main__":
    simple_test()

✅ 连接成功！
✅ 初始化成功！
📨 服务器响应: {"uid": "test", "code": 1002, "status": "Version Error", "message": "api version is error\uff01"}
✅ 测试完成！


In [12]:
import websocket._core as websocket
import json
import soundfile as sf
import numpy as np
import time
def quick_test():
        ws = websocket.create_connection("ws://127.0.0.1:9092")
        print("✅ WebSocket connection established!")
        
        # Correct initialization message
        init_msg = {
            "uid": "test_user_123",
            "token": "test_token",
            "name": "test_audio.wav", 
            "version": "1.0",  # 🔑 Required version number
            "model": "faster_whisper",
            "initial_prompt": "",
            "user_id": "test123",
            "type_name": "developer"
        }
        
        print("📤 Sending initialization message...")
        ws.send(json.dumps(init_msg))
        
        # === 3. Load audio and convert to float32 ===
        wav_path = "/workspace/ASR/audio_examples/self_record51.wav"
        print(f"Loading audio from {wav_path}")
        audio_data, sample_rate = sf.read(wav_path, dtype='float32')
        assert sample_rate == 16000, "Audio sample rate must be 16kHz"

        # === 4. Convert to int16 PCM (send in frames) ===
        int16_audio = (audio_data * 32768.0).astype(np.int16)
        frame_duration = 3  # Each frame is 0.4 seconds, adjustable
        frame_size = int(sample_rate * frame_duration)  
        num_frames = len(int16_audio) // frame_size

        print(f"Sending audio in {num_frames+1} frames...")
        #ws.send(int16_audio.tobytes(), opcode=websocket.ABNF.OPCODE_BINARY)
        
        for i in range(num_frames + 1):
            start = i * frame_size
            end = start + frame_size
            frame = int16_audio[start:end]
            if len(frame) == 0:
                continue
            ws.send(frame.tobytes(), opcode=websocket.ABNF.OPCODE_BINARY)
            time.sleep(frame_duration)  # Simulate real-time sending

        # === 5. Send end-of-audio flag ===
        ws.send(b"END_OF_AUDIO")
        print("Sent END_OF_AUDIO.")

        # === 6. Receive and print returned content ===
        print("Receiving results...")
        while True:
            try:
                msg = ws.recv()
                if not msg:
                    break
                print("Received response:", msg)
                if '"is_end": true' in msg:
                    print("✅ Server recognition finished")
                    break
            except Exception as e:
                print(f"Connection closed or exception: {e}")
                break
        
        # === 7. Close connection ===
        ws.close()
        print("WebSocket closed.")


quick_test()


✅ WebSocket connection established!
📤 Sending initialization message...
Loading audio from /workspace/ASR/audio_examples/self_record51.wav
Sending audio in 10 frames...


Sent END_OF_AUDIO.
Receiving results...
Received response: {"uid": "test_user_123", "message": "SERVER_READY", "code": 0, "status": "SERVER_READY", "backend": "vvVoiceServer"}
Received response: {"uid": "test_user_123", "code": 0, "status": "RESULT", "segments": [{"start": "0.000", "end": "3.000", "text": " I spent the entire afternoon working on that report."}], "is_end": false}
Received response: {"uid": "test_user_123", "code": 0, "status": "RESULT", "segments": [{"start": "0.000", "end": "6.000", "text": " I spent the entire afternoon working on that report, only for my manager to say its not quite what."}], "is_end": false}
Received response: {"uid": "test_user_123", "code": 0, "status": "RESULT", "segments": [{"start": "0.000", "end": "9.000", "text": " I spent the entire afternoon working on that report, only for my manager to say its not quite what they expected. They didnt even give clear."}], "is_end": false}
Received response: {"uid": "test_user_123", "code": 0, "status": "R

In [17]:
import websocket._core as websocket
import json
import soundfile as sf
import numpy as np
import time

def quick_test():
        ws = websocket.create_connection("ws://127.0.0.1:9092")
        print("✅ WebSocket connection established!")
        
        init_msg = {
            "uid": "test_user_123",
            "token": "test_token",
            "name": "test_audio.wav", 
            "version": "1.0",
            "model": "faster_whisper",
            "initial_prompt": "",
            "user_id": "test123",
            "type_name": "developer"
        }
        print("📤 Sending initialization message...")
        ws.send(json.dumps(init_msg))
        
        # 读取音频
        wav_path = "/workspace/ASR/audio_examples/self_record51.wav"
        print(f"Loading audio from {wav_path}")
        audio_data, sample_rate = sf.read(wav_path, dtype='float32')
        assert sample_rate == 16000, "Audio sample rate must be 16kHz"

        # 转int16分帧发送
        int16_audio = (audio_data * 32768.0).astype(np.int16)
        frame_duration = 3
        frame_size = int(sample_rate * frame_duration)  
        num_frames = len(int16_audio) // frame_size-4

        print(f"Sending audio in {num_frames+1} frames...")
        end_flag = False

        for i in range(num_frames + 1):
            start = i * frame_size
            end = start + frame_size
            frame = int16_audio[start:end]
            if len(frame) == 0:
                continue

            # 发送当前帧
            ws.send(frame.tobytes(), opcode=websocket.ABNF.OPCODE_BINARY)

            # 发送后立刻尽量接收（短超时，拉取可用结果）
            ws.settimeout(0.2)
            while True:
                try:
                    msg = ws.recv()
                    if not msg:
                        break
                    print("Received response:", msg)
                    if '"is_end": true' in msg:
                        end_flag = True
                        break
                except Exception:
                    break  # 短超时或无消息立即返回
            if end_flag:
                break

            # 再sleep以模拟实时
            time.sleep(frame_duration)

        if not end_flag:
            # 发送结束标志
            ws.send(b"END_OF_AUDIO")
            print("Sent END_OF_AUDIO.")

            # 等待最终结果（更长超时）
            ws.settimeout(5)
            while True:
                try:
                    msg = ws.recv()
                    if not msg:
                        break
                    print("Received response:", msg)
                    if '"is_end": true' in msg:
                        print("✅ Server recognition finished")
                        break
                except Exception as e:
                    print(f"Connection closed or exception: {e}")
                    break
        
        ws.close()
        print("WebSocket closed.")

quick_test()

✅ WebSocket connection established!
📤 Sending initialization message...
Loading audio from /workspace/ASR/audio_examples/self_record51.wav
Sending audio in 6 frames...
Received response: {"uid": "test_user_123", "message": "SERVER_READY", "code": 0, "status": "SERVER_READY", "backend": "vvVoiceServer"}
Received response: {"uid": "test_user_123", "code": 0, "status": "RESULT", "segments": [{"start": "0.000", "end": "3.000", "text": " I spent the entire afternoon working on that report."}], "is_end": false}
Received response: {"uid": "test_user_123", "code": 0, "status": "RESULT", "segments": [{"start": "0.000", "end": "3.000", "text": " I spent the entire afternoon working on that report."}, {"start": "3.000", "end": "6.000", "text": " Only for my manager to say its not quite what."}], "is_end": false}
Received response: {"uid": "test_user_123", "code": 0, "status": "RESULT", "segments": [{"start": "0.000", "end": "3.000", "text": " I spent the entire afternoon working on that report."}